# Combined Deepfake Detection: Attention + Feature Selection

### Cell 1: Check Data Directories
This step remains the same.

In [1]:
!ls /kaggle/input/1000-videos-split/1000_videos/train/fake | head -n 5
!ls /kaggle/input/1000-videos-split/1000_videos/train/real | head -n 5

128_896_10.png
128_896_11.png
128_896_12.png
128_896_13.png
128_896_14.png
ls: write error: Broken pipe
129_10.png
129_2.png
129_3.png
129_4.png
129_5.png
ls: write error: Broken pipe


### Hardware Detection

### Cell 2: Code (Check Data & Setup)

This cell runs a check to identify the available hardware and selects the appropriate TensorFlow distribution strategy.

In [2]:
import os
import tensorflow as tf

# Define your data path based on your Kaggle dataset
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

# Check if the directories exist
print("--- Checking Data Paths ---")
print(f"Train Real exists: {os.path.exists(os.path.join(DATA_PATH, 'train', 'real'))}")
print(f"Train Fake exists: {os.path.exists(os.path.join(DATA_PATH, 'train', 'fake'))}")
print(f"Val Real exists:   {os.path.exists(os.path.join(DATA_PATH, 'validation', 'real'))}")
print(f"Val Fake exists:   {os.path.exists(os.path.join(DATA_PATH, 'validation', 'fake'))}")
print(f"Test Real exists:  {os.path.exists(os.path.join(DATA_PATH, 'test', 'real'))}")
print(f"Test Fake exists:  {os.path.exists(os.path.join(DATA_PATH, 'test', 'fake'))}")
print("---------------------------")

def get_distribution_strategy():
    """
    Detects available hardware (TPU, multi-GPU, single-GPU, CPU) and returns
    the appropriate TensorFlow distribution strategy.
    """
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
        strategy = tf.distribute.TPUStrategy(tpu)
        print("✅ Running on TPU")
    except (ValueError, tf.errors.NotFoundError):
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 1:
            strategy = tf.distribute.MirroredStrategy()
            print(f"✅ Running on {len(gpus)} GPUs")
        elif len(gpus) == 1:
            strategy = tf.distribute.get_strategy()
            print("✅ Running on a single GPU")
        else:
            strategy = tf.distribute.get_strategy()
            print("✅ Running on CPU")
            
    print(f"Number of accelerator replicas: {strategy.num_replicas_in_sync}")
    return strategy
    
# Run the detection function to see what hardware is available
strategy = get_distribution_strategy()

2025-11-12 12:38:30.923998: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762951111.156112      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762951111.219577      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

--- Checking Data Paths ---
Train Real exists: True
Train Fake exists: True
Val Real exists:   True
Val Fake exists:   True
Test Real exists:  True
Test Fake exists:  True
---------------------------
✅ Running on a single GPU
Number of accelerator replicas: 1


**MODEL DEFINITION**
## Part 1: Write Python Modules

These cells will write the project's logic into separate `.py` files in the environment. This keeps the project modular, just like your example.

In [3]:
%%writefile model.py
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    BatchNormalization,
    Dropout,
    Reshape,
    Add,
    Flatten,
)
from tensorflow.keras.models import Model
import tensorflow.keras.applications.densenet as densenet
import tensorflow.keras.applications.efficientnet as efficientnet
import tensorflow.keras.applications.xception as xception

# --- Custom Attention Layers (from 2022 paper) ---

class ModifiedBranch(tf.keras.layers.Layer):
    """
    Computes the modified branch for the attention mechanism.
    (From deepfake-detection2/model.py)
    """
    def __init__(self, a_vec_size, **kwargs):
        super(ModifiedBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
        self.dense_layer = Dense(self.a_vec_size, activation="tanh", name="att_mod_dense")

    def call(self, input):
        af = tf.keras.backend.mean(input, axis=2)
        hs = self.dense_layer(af)
        return hs

    def get_config(self):
        config = super().get_config()
        config.update({"a_vec_size": self.a_vec_size})
        return config


class MainBranch(tf.keras.layers.Layer):
    """
    Computes the main branch for the attention mechanism.
    (From deepfake-detection2/model.py)
    """
    def __init__(self, a_vec_size, dim, **kwargs):
        super(MainBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
        self.dim = dim
        self.reshape1 = Reshape((-1, self.a_vec_size), name="att_main_reshape1")
        self.relu = tf.keras.activations.relu
        self.dropout = Dropout(0.5, name="att_main_dropout")
        self.reshape2 = Reshape((self.dim**2, self.a_vec_size), name="att_main_reshape2")

    def call(self, input):
        e = tf.transpose(input, perm=[0, 2, 1])
        e = self.reshape1(e)
        e = self.relu(e)
        e = self.dropout(e)
        e = self.reshape2(e)
        e = tf.transpose(e, perm=[0, 2, 1])
        return e

    def get_config(self):
        config = super().get_config()
        config.update({"a_vec_size": self.a_vec_size, "dim": self.dim})
        return config


class Attention(tf.keras.layers.Layer):
    """
    Implements the attention technique on the two branches.
    (From deepfake-detection2/model.py)
    """
    def __init__(self, dim, a_vec_size, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.dim = dim
        self.a_vec_size = a_vec_size
        self.dense1 = Dense(self.dim**2, name="att_att_dense1")
        self.reshape1 = Reshape((1, self.dim**2), name="att_att_reshape1")
        self.add = Add(name="att_att_add")
        self.dropout = Dropout(0.5, name="att_att_dropout")
        self.relu = tf.keras.activations.relu
        self.reshape2 = Reshape((-1, self.a_vec_size), name="att_att_reshape2")
        self.dense2 = Dense(1, use_bias=False, name="att_att_dense2")
        self.reshape3 = Reshape((-1, self.dim**2), name="att_att_reshape3")

    def call(self, input):
        # input[0] is modified branch, input[1] is main branch
        eh = self.dense1(input[0])
        eh = self.reshape1(eh)
        eh = self.add([input[1], eh])
        eh = self.relu(eh)
        eh = self.dropout(eh)
        eh = tf.transpose(eh, perm=[0, 2, 1])
        eh = self.reshape2(eh)
        eh = self.dense2(eh)
        eh = self.reshape3(eh)
        eh = self.relu(eh)
        # The output 'eh' is the attention-weighted feature map (flattened)
        # We return the features *before* the final classification layer.
        return eh

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "a_vec_size": self.a_vec_size})
        return config

# --- Backbone Configuration and Extractor Creation (Combined Logic) ---

def get_backbone_config(backbone_name, input_shape=(299, 299, 3)):
    """
    Returns the correct pre-trained backbone, preprocessing function,
    feature map layer, and dimensions.
    """
    if backbone_name == "DenseNet121":
        base_model = densenet.DenseNet121(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        # Output map is (None, 9, 9, 1024)
        feature_map_layer = base_model.layers[-2].output
        preprocess_func = densenet.preprocess_input
        dim = 9
        a_vec_size = 1024
    elif backbone_name == "EfficientNetB0":
        base_model = efficientnet.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        # Output map is (None, 10, 10, 1280) for 299 input
        feature_map_layer = base_model.layers[-3].output
        preprocess_func = efficientnet.preprocess_input
        dim = 10  # <-- THIS IS THE FIX (was 9)
        a_vec_size = 1280
    elif backbone_name == "Xception":
        base_model = xception.Xception(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        # Using layer -13 from the 2022 paper's notebook
        feature_map_layer = base_model.layers[-13].output
        preprocess_func = xception.preprocess_input
        dim = 19
        a_vec_size = 1024
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    base_model.trainable = False
    return base_model, preprocess_func, feature_map_layer, dim, a_vec_size

def create_attention_extractor(backbone_name, input_shape=(299, 299, 3)):
    """
    Creates a model that applies the attention mechanism to the feature maps.
    """
    base_model, _, feature_map_layer, dim, a_vec_size = get_backbone_config(
        backbone_name, input_shape
    )

    # This logic is from the `model` function in deepfake-detection2/model.py
    # We apply it to the feature map layer of the backbone
    # Input shape here is (None, dim, dim, a_vec_size)
    x = Conv2D(
        filters=a_vec_size,
        kernel_size=(1, 1),
        strides=(1, 1),
        padding="valid",
        use_bias=True,
        name=f"{backbone_name}_att_conv",
    )(feature_map_layer)
    x = BatchNormalization(axis=-1, name=f"{backbone_name}_att_bn")(x)
    x = tf.keras.activations.relu(x)
    x = Dropout(0.8, name=f"{backbone_name}_att_drop")(x)
    
    # Reshape to (None, a_vec_size, dim*dim)
    # e.g., for EfficientNetB0: (None, 1280, 10*10) -> (None, 1280, 100)
    x = Reshape((a_vec_size, dim**2), name=f"{backbone_name}_att_reshape")(x)

    # Apply the custom attention layers
    modified = ModifiedBranch(a_vec_size, name=f"{backbone_name}_mod_branch")(x)
    main = MainBranch(a_vec_size, dim, name=f"{backbone_name}_main_branch")(x)
    attention_features = Attention(dim, a_vec_size, name=f"{backbone_name}_attention")(
        [modified, main]
    )
    
    # Flatten the attention output to get our 1D feature vector
    output_features = Flatten(name=f"{backbone_name}_flatten")(attention_features)

    # Create the final extractor model
    extractor = Model(
        inputs=base_model.input,
        outputs=output_features,
        name=f"{backbone_name}_AttentionExtractor",
    )
    return extractor

print("model.py written.")


Writing model.py


**UTILITIES & FEATURES SELECTION**

In [4]:
%%writefile utils.py
import os
import glob
import numpy as np
import tensorflow.keras.preprocessing.image as tf_image
from sklearn.utils import shuffle
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras.applications.densenet as densenet
import tensorflow.keras.applications.efficientnet as efficientnet
import tensorflow.keras.applications.xception as xception

# --- Data Loading ---

def prepare_dataset_paths(base_path, train_counts, val_counts, test_counts):
    """Prepare dataset paths and labels based on subfolders."""
    print(f"Loading data paths from: {base_path}")
    
    def get_paths_labels(split, real_count, fake_count):
        # FIX: Use 'validation' folder name for val split, as per your dataset
        split_name = 'validation' if split == 'val' else split
        
        real_paths = sorted(glob.glob(os.path.join(base_path, split_name, "real", "*.*")))
        fake_paths = sorted(glob.glob(os.path.join(base_path, split_name, "fake", "*.*")))
        
        # Use provided counts to slice the lists
        if real_count > 0:
            real_paths = real_paths[:real_count]
        if fake_count > 0:
            fake_paths = fake_paths[:fake_count]

        paths = real_paths + fake_paths
        # Use 0 for REAL, 1 for FAKE
        labels = [0] * len(real_paths) + [1] * len(fake_paths)
        
        if not paths:
            print(f"  Loaded {split}: 0 paths.")
            return [], []

        paths, labels = shuffle(paths, labels, random_state=42)
        print(f"  Loaded {split}: {len(real_paths)} real, {len(fake_paths)} fake. Total: {len(paths)}")
        return paths, labels

    data = {}
    if train_counts:
        data["train"] = get_paths_labels("train", train_counts.get('real', 0), train_counts.get('fake', 0))
    if val_counts:
        data["val"] = get_paths_labels("val", val_counts.get('real', 0), val_counts.get('fake', 0))
    if test_counts:
        data["test"] = get_paths_labels("test", test_counts.get('real', 0), test_counts.get('fake', 0))
    
    return data


# --- Feature Extraction ---

def get_preprocessors():
    """Returns a dict of preprocessing functions for each model."""
    return {
        "DenseNet121": densenet.preprocess_input,
        "EfficientNetB0": efficientnet.preprocess_input,
        "Xception": xception.preprocess_input,
    }

def extract_features(file_paths, extractors, preprocessors, input_shape=(299, 299, 3), batch_size=32):
    """
    Extracts features from all three attention-based extractors and stacks them.
    Handles different preprocessing for each model.
    """
    all_features = {name: [] for name in extractors.keys()}
    total_files = len(file_paths)

    for i in range(0, total_files, batch_size):
        if i % (batch_size * 10) == 0:
            print(f"  Processing batch {i // batch_size} / {int(np.ceil(total_files / batch_size))}")
            
        batch_paths = file_paths[i : i + batch_size]
        loaded_images = []
        
        # Store images once
        for img_path in batch_paths:
            try:
                image = tf_image.load_img(
                    img_path, target_size=input_shape[:2]
                )
                image = tf_image.img_to_array(image)
                loaded_images.append(image)
            except Exception as e:
                print(f"Warning: Error loading {img_path}: {e}")
                # Add a dummy image to keep batch size consistent for indices
                loaded_images.append(np.zeros(input_shape))
                
        if not loaded_images:
            continue

        loaded_images_np = np.array(loaded_images)

        # Preprocess and extract for each model
        for name, extractor in extractors.items():
            preprocess_func = preprocessors[name]
            # Preprocess the batch
            preprocessed_batch = preprocess_func(loaded_images_np.copy())
            
            # Get features
            features = extractor.predict(preprocessed_batch, verbose=0)
            all_features[name].extend(features)

    print("Stacking extracted features...")
    stacked_features = np.concatenate(
        [
            np.array(all_features["DenseNet121"]),
            np.array(all_features["EfficientNetB0"]),
            np.array(all_features["Xception"]),
        ],
        axis=1,
    )
    return stacked_features


# --- Feature Selection (from 2025 paper) ---

def relief_f_score(X, y, k=10):
    """ReliefF feature selection algorithm"""
    print("  Running ReliefF...")
    n_samples, n_features = X.shape
    feature_scores = np.zeros(n_features)

    for i in range(n_samples):
        if i % 100 == 0:
            print(f"    ReliefF processing sample {i} / {n_samples}")
        distances = np.sum((X - X[i]) ** 2, axis=1)
        nearest_indices = np.argsort(distances)[1 : k + 1]
        hits = [idx for idx in nearest_indices if y[idx] == y[i]]
        misses = [idx for idx in nearest_indices if y[idx] != y[i]]

        for j in range(n_features):
            if hits:
                hit_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in hits])
                feature_scores[j] -= hit_diff
            if misses:
                miss_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in misses])
                feature_scores[j] += miss_diff
    
    min_score = np.min(feature_scores)
    max_score = np.max(feature_scores)
    if max_score == min_score:
        return np.zeros_like(feature_scores)
    return (feature_scores - min_score) / (max_score - min_score)

def mrmr_score(X, y, selected_features=None):
    """Minimum Redundancy Maximum Relevance score helper"""
    if selected_features is None:
        selected_features = []

    mi_scores = mutual_info_classif(X, y, random_state=42)

    if not selected_features:
        return mi_scores

    n_features = X.shape[1]
    mrmr_scores = np.zeros(n_features)

    for i in range(n_features):
        if i in selected_features:
            mrmr_scores[i] = -np.inf
            continue
        
        relevance = mi_scores[i]
        
        if selected_features:
            redundancy = np.mean(
                [
                    mutual_info_classif(X[:, [i, j]], y, random_state=42)[0]
                    for j in selected_features
                ]
            )
        else:
            redundancy = 0
        
        mrmr_scores[i] = relevance - redundancy
    return mrmr_scores

def calculate_fitness(X_train, y_train, X_val, y_val, feature_indices, weight=0.9):
    """Calculate fitness function for feature selection"""
    if len(feature_indices) == 0:
        return 0
    
    # Ensure feature_indices are integers for slicing
    feature_indices = np.array(feature_indices).astype(int)

    X_train_sel = X_train[:, feature_indices]
    X_val_sel = X_val[:, feature_indices]
    
    temp_scaler = StandardScaler()
    X_train_scaled = temp_scaler.fit_transform(X_train_sel)
    X_val_scaled = temp_scaler.transform(X_val_sel)
    
    temp_knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1) # Use all cores
    temp_knn.fit(X_train_scaled, y_train)
    accuracy = temp_knn.score(X_val_scaled, y_val)
    
    feature_ratio = len(feature_indices) / X_train.shape[1]
    fitness = weight * accuracy + (1 - weight) * (1 - feature_ratio)
    return fitness

def feature_selection(
    X_train, y_train, X_val, y_val, tau=0.3, alpha=0.1, beta=0.1, max_iterations=250
):
    """Combined feature selection with ReliefF, MI, mRMR and inclusion-exclusion"""
    print("Starting feature selection...")
    
    relief_scores = relief_f_score(X_train, y_train)
    
    print("  Running Mutual Information...")
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    print("  Running mRMR...")
    mrmr_scores = mrmr_score(X_train, y_train)

    # Normalize scores
    mi_scores = (mi_scores - np.min(mi_scores)) / (np.max(mi_scores) - np.min(mi_scores) + 1e-6)
    mrmr_scores = (mrmr_scores - np.min(mrmr_scores)) / (np.max(mrmr_scores) - np.min(mrmr_scores) + 1e-6)

    combined_scores = (relief_scores + mi_scores + mrmr_scores) / 3
    
    n_features = len(combined_scores)
    n_initial = int(tau * n_features)
    initial_indices = np.argsort(combined_scores)[-n_initial:]

    print(f"  Initial selection: {len(initial_indices)} features")
    
    best_features = initial_indices.copy()
    best_fitness = calculate_fitness(X_train, y_train, X_val, y_val, best_features)
    print(f"  Initial fitness: {best_fitness:.4f}")

    for iteration in range(max_iterations):
        current_features = best_features.copy()
        
        n_exclude = max(1, int(alpha * len(current_features)))
        if len(current_features) > n_exclude:
            exclude_indices = np.random.choice(
                len(current_features), n_exclude, replace=False
            )
            current_features = np.delete(current_features, exclude_indices)
        
        remaining_features = np.setdiff1d(np.arange(n_features), current_features)
        if len(remaining_features) > 0:
            n_include = min(int(beta * n_features), len(remaining_features))
            if n_include == 0: n_include = 1 # Ensure at least one
                
            remaining_scores = combined_scores[remaining_features]
            include_indices = remaining_features[
                np.argsort(remaining_scores)[-n_include:]
            ]
            current_features = np.unique(np.concatenate([current_features, include_indices]))
        
        current_fitness = calculate_fitness(X_train, y_train, X_val, y_val, current_features)

        if current_fitness > best_fitness:
            best_features = current_features.copy()
            best_fitness = current_fitness
            if (iteration + 1) % 10 == 0:
                print(
                    f"    Iter {iteration + 1}: New best fitness = {best_fitness:.4f}, Features = {len(best_features)}"
                )
    
    print(f"Feature selection completed. Selected {len(best_features)} features.")
    return best_features

print("utils.py written.")

Writing utils.py


**TRAINING SCRIPT**

In [5]:
%%writefile train.py
import os
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import gc

# We must import the custom layers for the model to load
import model
import utils

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

class train:
    def __init__(self, train_path, val_path, train_counts, val_counts):
        self.train_path = train_path
        self.val_path = val_path
        self.train_counts = train_counts
        self.val_counts = val_counts
        self.model_save_dir = "models"
        os.makedirs(self.model_save_dir, exist_ok=True)

    def run(self, tau, alpha, beta, max_iter, batch_size=32):
        print("--- Starting Training Pipeline ---")
        
        # 1. Load Data Paths
        train_data = utils.prepare_dataset_paths(
            self.train_path, self.train_counts, {}, {}
        )
        val_data = utils.prepare_dataset_paths(
            self.val_path, {}, self.val_counts, {}
        )
        
        train_paths, train_labels = train_data["train"]
        val_paths, val_labels = val_data["val"]

        # 2. Create Feature Extractors
        print("Loading feature extractor models...")
        model_names = ["DenseNet121", "EfficientNetB0", "Xception"]
        extractors = {}
        input_shape = (299, 299, 3)
        
        # Custom objects needed to load the Keras models
        custom_objects = {
            "ModifiedBranch": model.ModifiedBranch, 
            "MainBranch": model.MainBranch, 
            "Attention": model.Attention
        }
    
        with tf.keras.utils.custom_object_scope(custom_objects):
            for name in model_names:
                print(f"  Creating {name} extractor...")
                extractors[name] = model.create_attention_extractor(name, input_shape)
        
        preprocessors = utils.get_preprocessors()
        
        # 3. Extract Features
        print("\nExtracting Training Features... (This may take a while)")
        train_features = utils.extract_features(
            train_paths, extractors, preprocessors, input_shape, batch_size
        )
        print("\nExtracting Validation Features...")
        val_features = utils.extract_features(
            val_paths, extractors, preprocessors, input_shape, batch_size
        )
        
        # Ensure labels are numpy arrays
        train_labels = np.array(train_labels)
        val_labels = np.array(val_labels)
        
        print(f"\nTotal stacked feature dimension: {train_features.shape[1]}")
        
        # --- Memory Cleanup ---
        # We are done with the large Keras models, release GPU memory
        del extractors
        del preprocessors
        tf.keras.backend.clear_session()
        gc.collect()
        print("GPU models cleared from memory.")
        # --- End Memory Cleanup ---

        # 4. Run Feature Selection
        selected_indices = utils.feature_selection(
            train_features, train_labels, 
            val_features, val_labels, 
            tau, alpha, beta, max_iter
        )

        # 5. Train Final Classifier
        print("Training final KNN classifier...")
        knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1) # Use all cores
        scaler = StandardScaler()
        
        X_train_selected = train_features[:, selected_indices.astype(int)]
        X_train_scaled = scaler.fit_transform(X_train_selected)
        knn.fit(X_train_scaled, train_labels)

        # 6. Save models to disk
        print(f"Saving models to '{self.model_save_dir}' directory...")
        joblib.dump(knn, os.path.join(self.model_save_dir, 'knn_model.joblib'))
        joblib.dump(scaler, os.path.join(self.model_save_dir, 'scaler.joblib'))
        joblib.dump(selected_indices, os.path.join(self.model_save_dir, 'selected_features.joblib'))
        
        print("--- Training Pipeline Complete ---")

print("train.py written.")


Writing train.py


**MAIN ENTRY POINT**

In [6]:
%%writefile main.py
import argparse
from train import train
import json

def main():
    parser = argparse.ArgumentParser(description='Train Combined Deepfake Detector')

    # Paths
    parser.add_argument('--train_path', type=str, required=True, help='Base path to training dataset')
    parser.add_argument('--val_path', type=str, required=True, help='Base path to validation dataset')

    # Data counts
    # Updated defaults based on your dataset
    parser.add_argument('--train_real', type=int, default=5605, help='Num real training images')
    parser.add_argument('--train_fake', type=int, default=6028, help='Num fake training images')
    parser.add_argument('--val_real', type=int, default=1200, help='Num real validation images')
    parser.add_argument('--val_fake', type=int, default=1200, help='Num fake validation images')

    # Feature selection params
    parser.add_argument('--tau', type=float, default=0.3, help='Initial selection threshold')
    parser.add_argument('--alpha', type=float, default=0.1, help='Exclusion percentage')
    parser.add_argument('--beta', type=float, default=0.1, help='Inclusion percentage')
    parser.add_argument('--max_iter', type=int, default=250, help='Optimization iterations')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for feature extraction')

    args = parser.parse_args()

    train_counts = {'real': args.train_real, 'fake': args.train_fake}
    val_counts = {'real': args.val_real, 'fake': args.val_fake}

    print("--- Configuration ---")
    print(f"Train Path: {args.train_path}")
    print(f"Val Path: {args.val_path}")
    print(f"Train Counts: {train_counts}")
    print(f"Val Counts: {val_counts}")
    print(f"FS Params: tau={args.tau}, alpha={args.alpha}, beta={args.beta}, iter={args.max_iter}")
    print("---------------------\n")

    trainer = train(args.train_path, args.val_path, train_counts, val_counts)
    trainer.run(args.tau, args.alpha, args.beta, args.max_iter, args.batch_size)

if __name__ == '__main__':
    main()
    
print("main.py written.")


Writing main.py


**PREDICTION SCRIPT**

In [7]:
%%writefile predict.py
import argparse
import joblib
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# We must import the custom layers for the model to load
import model
import utils

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

def main():
    parser = argparse.ArgumentParser(description='Evaluate Combined Deepfake Detector')
    
    # Paths
    parser.add_argument('--test_path', type=str, required=True, help='Base path to test dataset')
    parser.add_argument('--model_dir', type=str, default='models', help='Directory containing saved models')

    # Data counts
    # Updated defaults based on your dataset
    parser.add_argument('--test_real', type=int, default=1200, help='Num real test images')
    parser.add_argument('--test_fake', type=int, default=1200, help='Num fake test images')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for feature extraction')

    args = parser.parse_args()

    print("--- Starting Prediction Pipeline ---")

    # 1. Load saved models
    print(f"Loading models from {args.model_dir}...")
    try:
        knn = joblib.load(os.path.join(args.model_dir, 'knn_model.joblib'))
        scaler = joblib.load(os.path.join(args.model_dir, 'scaler.joblib'))
        selected_indices = joblib.load(os.path.join(args.model_dir, 'selected_features.joblib'))
    except FileNotFoundError as e:
        print(f"Error: Model file not found. {e}")
        print("Please run train.py first to generate the models.")
        return

    # 2. Create Feature Extractors
    print("Loading feature extractor models...")
    model_names = ["DenseNet121", "EfficientNetB0", "Xception"]
    extractors = {}
    input_shape = (299, 299, 3)
    
    # Custom objects needed to load the Keras models
    custom_objects = {
        "ModifiedBranch": model.ModifiedBranch, 
        "MainBranch": model.MainBranch, 
        "Attention": model.Attention
    }
    
    with tf.keras.utils.custom_object_scope(custom_objects):
        for name in model_names:
            extractors[name] = model.create_attention_extractor(name, input_shape)
    
    preprocessors = utils.get_preprocessors()

    # 3. Load Test Data Paths
    test_counts = {'real': args.test_real, 'fake': args.test_fake}
    test_data = utils.prepare_dataset_paths(
        args.test_path, {}, {}, test_counts
    )
    test_paths, test_labels = test_data["test"]
    if not test_paths:
        print(f"No test images found in {args.test_path}. Exiting.")
        return

    # 4. Extract Test Features
    print("\nExtracting Test Features... (This may take a while)")
    test_features = utils.extract_features(
        test_paths, extractors, preprocessors, input_shape, args.batch_size
    )
    
    test_labels = np.array(test_labels)

    # --- Memory Cleanup ---
    del extractors
    del preprocessors
    tf.keras.backend.clear_session()
    gc.collect()
    print("GPU models cleared from memory.")
    # --- End Memory Cleanup ---

    # 5. Apply Selection and Scaling
    print("Applying feature selection and scaling...")
    X_test_selected = test_features[:, selected_indices.astype(int)]
    X_test_scaled = scaler.transform(X_test_selected)

    # 6. Run Prediction
    print("Running predictions...")
    predictions = knn.predict(X_test_scaled)
    probabilities = knn.predict_proba(X_test_scaled)

    # 7. Report Results
    test_accuracy = accuracy_score(test_labels, predictions)
    test_auc = roc_auc_score(test_labels, probabilities[:, 1])
    cm = confusion_matrix(test_labels, predictions)

    print("\n--- FINAL TEST RESULTS ---")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    print(f"  Original features: {test_features.shape[1]}")
    print(f"  Selected features: {len(selected_indices)}")
    print(f"  Feature reduction: {(1 - len(selected_indices)/test_features.shape[1])*100:.2f}%")
    print("  Confusion Matrix:")
    print(cm)
    print("-------------------------")

    # Plot and save confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Test Set Confusion Matrix')
    plt.savefig(os.path.join(args.model_dir, 'confusion_matrix.png'))
    print(f"Confusion matrix saved to {args.model_dir}/confusion_matrix.png")

if __name__ == '__main__':
    main()

print("predict.py written.")

Writing predict.py


## Part 2: Train the Model

This cell executes the `main.py` script to start the training process. All your dataset counts are now set as defaults inside `main.py`.

**This is a very long-running cell.** It will:
1.  Extract features from all 3 models for all **11,633 train** images.
2.  Extract features from all 3 models for all **2,400 validation** images.
3.  Run the computationally expensive feature selection algorithms (ReliefF, MI, mRMR).
4.  Run the 250-iteration inclusion-exclusion optimization.
5.  Train and save the final KNN model, scaler, and feature list to the `models/` directory.

In [8]:
# Set the data path variable
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

print("--- [Step 1] Running Training ---")

# Run main.py. It will use the default counts you provided.
# We pass the paths as arguments.
# We also set a smaller batch size to help with memory on the GPU.
!python main.py \
    --train_path "$DATA_PATH" \
    --val_path "$DATA_PATH" \
    --batch_size 16

--- [Step 1] Running Training ---
2025-11-12 12:38:48.107897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762951128.130170     104 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762951128.136662     104 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
model.py written.
utils.py written.
train.p

## Part 3: Run Prediction

After training is complete, this cell runs `predict.py`. It will load the saved models (`knn_model.joblib`, etc.) from the `models/` folder and evaluate them on your test set.

In [9]:
# Set the data path variable
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

print("--- [Step 2] Running Prediction ---")

# Run predict.py. It will use the default test counts (1200 real, 1200 fake).
!python predict.py \
    --test_path "$DATA_PATH" \
    --batch_size 16

--- [Step 2] Running Prediction ---
2025-11-12 12:52:57.903227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762951977.924845   32320 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762951977.931306   32320 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
model.py written.
utils.py written.
--- S